# 🔬 SciMSim — Scientific Scenario Simulation

A modular LLM-driven simulation engine for exploring hypothetical futures of scientific fields.

**How it works:**
- A population of researcher *personas* is generated from your config.
- Each timestep, active researchers read papers from the growing corpus and write new ones.
- Papers cite prior work; impact scores propagate through the corpus.
- External shocks can be injected at any timestep.
- After the run, you get a readable timeline, top papers, and keyword evolution.

This is **scenario exploration**, not prediction — the goal is to generate internally consistent,
plausible alternative futures and stress-test assumptions.

## 0. Install & Import

In [1]:
# Install dependencies if needed
!pip install anthropic          # for Claude backend
# !pip install openai             # for GPT backend

!git clone https://github.com/nglguarino/scimsim.git
%cd scimsim/notebooks

import sys, os
sys.path.insert(0, os.path.abspath('..'))

from scimsim import (
    SimulationConfig,
    ExternalShock,
    SchoolOfThought,
    IncentiveStructure,
    run_simulation,
    print_scenario_summary,
    timeline_view,
    export_corpus_md,
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 42.2 MB/s eta 0:00:00
Cloning into 'scimsim'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 17 (delta 2), reused 17 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 17.75 KiB | 8.88 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/scimsim/notebooks


## 1. API Key

Set your API key here. It is never stored or logged.

In [2]:
# Choose your provider
PROVIDER = "anthropic"   # or "openai"

# Paste your key here, or load from environment, or from Colab Secrets
#API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
# API_KEY = os.environ.get("OPENAI_API_KEY", "")  # if using OpenAI

#if not API_KEY:
    #API_KEY = input("Paste your API key: ").strip()  # to type the key in

from google.colab import userdata
API_KEY = userdata.get("ANTHROPIC_API_KEY")

print(f"Provider: {PROVIDER} | Key set: {'yes' if API_KEY else 'NO'}")

Provider: anthropic | Key set: yes


## 2. Configure Your Simulation

All parameters are here. Change them freely — each combination produces a different scenario.

| Parameter | What it controls |
|-----------|------------------|
| `domain` | The scientific field being simulated |
| `seed_idea` | The founding idea the simulation bootstraps from |
| `num_timesteps` | How many years to simulate |
| `papers_per_timestep` | Research output per year (more = richer but slower) |
| `num_researchers` | Size of the simulated community |
| `epistemic_openness` | 0 = paradigm-locked, 1 = freely cross-pollinating |
| `incentive_structure` | academic / industry / balanced |
| `wildcard` | 0 = conservative ideas, 1 = weird/unexpected |
| `shocks` | Discrete events injected at specific timesteps |

In [4]:
cfg = SimulationConfig(
    # ── Core setup ─────────────────────────────────────────────────────
    domain       = "artificial intelligence",
    seed_idea    = "large language models can reason across domains when scaled",
    start_year   = 2025,
    num_timesteps       = 5,   # simulated years
    papers_per_timestep = 3,   # papers generated per year

    # ── Community sociology ────────────────────────────────────────────
    num_researchers = 12,
    school_distribution = {
        SchoolOfThought.DOMINANT:    0.4,   # mainstream scaling researchers
        SchoolOfThought.CHALLENGER:  0.2,   # heterodox critics
        SchoolOfThought.APPLIED:     0.3,   # engineers / deployment focus
        SchoolOfThought.THEORETICAL: 0.1,   # formal / mathematical
    },
    epistemic_openness = 0.5,   # 0=stay in paradigm, 1=freely cross

    # ── Incentives ─────────────────────────────────────────────────────
    incentive_structure = IncentiveStructure.BALANCED,

    # ── Wildcards ──────────────────────────────────────────────────────
    wildcard = 0.35,   # 0=conservative, 1=unexpected ideas

    # ── External shocks ────────────────────────────────────────────────
    # Inject discrete events at specific timesteps (0-indexed).
    # Comment out to run without shocks.
    shocks = [
        ExternalShock(
            timestep=2,
            description=(
                "A major safety incident involving a deployed LLM causes "
                "public backlash and forces the community to prioritize "
                "alignment and interpretability research."
            ),
        ),
        ExternalShock(
            timestep=4,
            description=(
                "Compute costs drop 10x due to new hardware. "
                "Small labs can now run experiments previously only possible at frontier labs."
            ),
            affects_schools=[SchoolOfThought.CHALLENGER, SchoolOfThought.APPLIED],
        ),
    ],

    # ── LLM backend ────────────────────────────────────────────────────
    llm_provider = PROVIDER,
    api_key      = API_KEY,
    llm_model  = "claude-opus-4-7"  # e.g. "claude-haiku-4-5", "claude-sonnet-4-6"
)

## 3. Run the Simulation

In [5]:
state = run_simulation(cfg, verbose=True)


🔬 SciMSim — Scientific Scenario Simulation
   Provider : anthropic  |  Model: claude-opus-4-7
   Domain   : artificial intelligence
   Timesteps: 5  |  Papers/step: 3
   Wildcard : 0.35  |  Openness: 0.5

👥 Generating 12 researchers...
   • Daniel Kaufman [dominant] — scaling laws, emergent capabilities
   • Wei-Lin Chen [dominant] — chain-of-thought reasoning, instruction tuning
   • Priya Venkatesan [dominant] — multi-task generalization, pretraining data curation
   • Aleksandr Volkov [dominant] — mixture-of-experts, efficient inference
   • Sofia Marchetti [dominant] — in-context learning, prompt engineering
   • Ngozi Adebayo [challenger] — compositional generalization critique, symbolic-neural hybrids
   • Finn O'Sullivan [challenger] — stochastic parrots debate, memorization vs reasoning
   • Rajesh Krishnamurthy [applied] — MMLU benchmarks, code generation
   • Emma Thorsen [applied] — medical QA systems, RAG pipelines
   • Carlos Mendoza [applied] — agent frameworks, tool use

## 4. Explore the Results

In [6]:
# High-level stats and top papers
print_scenario_summary(state)


  SIMULATION COMPLETE — SCENARIO SUMMARY
  Domain        : artificial intelligence
  Seed idea     : large language models can reason across domains when scaled
  Years covered : 2025 – 2029
  Total papers  : 16
  Researchers   : 12

  Papers by school of thought:
    dominant     ████████ (8)
    applied      ████ (4)
    challenger   ███ (3)
    theoretical  █ (1)

  Top 5 papers by impact score:
    1. [9.0] Foundations of Artificial Intelligence: Large language models can reason across domains when scaled
         Year 2024 · dominant · by [Seed]
    2. [6.8] CG-SERVE-Federated: Small-Lab Circuit Auditing and Site-Specific Library Extension for Clinical RAG under Commodity Compute
         Year 2029 · applied · by Emma Thorsen
    3. [6.5] The Universality Test at 10x Compute: Adversarial Cross-Site Replication of the 70-Feature Aligned-Circuit Library
         Year 2029 · dominant · by Daniel Kaufman
    4. [6.5] Circuit Libraries Are Not Enough: Temporal-Strategic Reasoning Gaps

In [7]:
# Readable year-by-year timeline with abstracts
timeline_view(state)


──────────────────────────────────────────────────────────────────────
  TIMELINE: ARTIFICIAL INTELLIGENCE
  Starting from: "large language models can reason across domains when scaled"
──────────────────────────────────────────────────────────────────────

  ── Year 2025 (t=1) ──
  📄 On the Expressivity-Generalization Gap in Scaled Transformers: An Information-Theoretic Critique of Cross-Domain Reasoning Claims
     Recent empirical work has claimed that large language models acquire cross-domain reasoning capabilities as a function of scale. In this paper, we interrogate this claim from a lea...
     [theoretical] · impact 5.13 · keywords: transformer expressivity, information-theoretic bounds, cross-domain generalization, learning theory

  📄 Memorization Masquerading as Reasoning: An Empirical Audit of Cross-Domain Benchmarks via Training Data Attribution
     Claims that large language models reason across domains rest almost entirely on aggregate benchmark performance, with litt

In [8]:
# Export the full corpus as a Markdown file
export_corpus_md(state, path="my_scenario.md")

  ✅ Corpus exported to my_scenario.md


## 5. Compare Two Scenarios

The real power of the engine is running the same seed under different parameters
and comparing where the field ends up.

In [11]:
# Scenario A: industry-driven, conservative, no shocks
cfg_a = SimulationConfig(
    domain="artificial intelligence",
    seed_idea="large language models can reason across domains when scaled",
    start_year=2025,
    num_timesteps=4,
    papers_per_timestep=2,
    incentive_structure=IncentiveStructure.INDUSTRY,
    wildcard=0.1,
    epistemic_openness=0.2,
    llm_provider=PROVIDER,
    api_key=API_KEY,
)

# Scenario B: academic, high wildcard, major shock
cfg_b = SimulationConfig(
    domain="artificial intelligence",
    seed_idea="large language models can reason across domains when scaled",
    start_year=2025,
    num_timesteps=4,
    papers_per_timestep=2,
    incentive_structure=IncentiveStructure.ACADEMIC,
    wildcard=0.8,
    epistemic_openness=0.9,
    shocks=[ExternalShock(timestep=1, description="A breakthrough in neuromorphic computing makes transformer architectures obsolete.")],
    llm_provider=PROVIDER,
    api_key=API_KEY,
)

print("Running Scenario A (industry, conservative)...")
state_a = run_simulation(cfg_a, verbose=False)

print("\nRunning Scenario B (academic, wild, shocked)...")
state_b = run_simulation(cfg_b, verbose=False)

print("\n" + "="*70)
print("SCENARIO A — Top papers:")
for p in sorted(state_a.corpus, key=lambda p: p.impact_score, reverse=True)[:3]:
    print(f"  [{p.impact_score:.1f}] {p.title}")

print("\nSCENARIO B — Top papers:")
for p in sorted(state_b.corpus, key=lambda p: p.impact_score, reverse=True)[:3]:
    print(f"  [{p.impact_score:.1f}] {p.title}")

Running Scenario A (industry, conservative)...

Running Scenario B (academic, wild, shocked)...

SCENARIO A — Top papers:
  [9.0] Foundations of Artificial Intelligence: Large language models can reason across domains when scaled
  [6.6] Mechanistic Degradation Under Scale: Why Larger Models Exhibit Weaker Circuit Interpretability in Cross-Domain Transfer
  [6.5] Circuit Alignment and Semantic Grounding: Mechanistic Conditions for Robust Multi-Domain Transfer Without Scale

SCENARIO B — Top papers:
  [9.0] Foundations of Artificial Intelligence: Large language models can reason across domains when scaled
  [7.3] Production-Grade Reasoning Verification: Decoupling Interpretability From Deployment in 2028 LLM Systems
  [7.0] The Embodied Reasoning Fallacy: Why Substrate Transfer Doesn't Solve the Compositional Poverty Problem in Neural Systems


## 6. Inspect Individual Papers

In [9]:
# Browse all papers from the main run
for p in state.corpus:
    if p.timestep < 0:
        continue  # skip seed
    print(f"Year {cfg.year_of(p.timestep)} | [{p.school.value}] | Impact {p.impact_score}")
    print(f"  {p.title}")
    print(f"  {p.abstract[:200]}...")
    print()

Year 2025 | [theoretical] | Impact 5.13
  On the Expressivity-Generalization Gap in Scaled Transformers: An Information-Theoretic Critique of Cross-Domain Reasoning Claims
  Recent empirical work has claimed that large language models acquire cross-domain reasoning capabilities as a function of scale. In this paper, we interrogate this claim from a learning-theoretic pers...

Year 2025 | [challenger] | Impact 5.54
  Memorization Masquerading as Reasoning: An Empirical Audit of Cross-Domain Benchmarks via Training Data Attribution
  Claims that large language models reason across domains rest almost entirely on aggregate benchmark performance, with little attention paid to what the model has actually seen during pretraining. We c...

Year 2025 | [dominant] | Impact 5.92
  Process-Level Supervision for Robust Cross-Domain Reasoning: Beyond Outcome-Based Contamination
  Recent critiques have argued that apparent cross-domain reasoning in large language models reflects memorization over de